<br> 

***  

# <center>**Notebook 06 – XGBoost Hyperparameter Tuning**</center>

<br>

<center>Supporting materials for</center>

### <center>MSc Data Science</center>
## <center>Module: DSM500 Final Project</center>

<br>

<center>5 October 2026</center>

***
<br>

**Purpose of this notebook**  
This note book implements hyperparameter tuning for XGBoost for different scenarios. Results from tuning are included in this notebook and final selected parameter values are saved manually in a spreadsheet, as described in Notebook `07_XGB_Hyperparameter_Configuration.ipynb`. Temporary working files and the final spreadsheet are saved here:
```
..
    |-- scenarios
        |-- parameters
                |-- large-ir003.json
                |-- ...
                |-- small-ir081.json
    |-- models
        |-- xgboost
            |-- hyperparameters
                |-- hyperparameters.xlsx
```

## Overview of Approach

To make the grid search tractable, select the hyperparameters that are relatively uneffected by sample size and imbalance and tune those first. These are held constant for every scenario. Next, perform a grid search on the remaining parameters to find a centre point, then use this to define a range for grid search performed on every scenario. For both types of hyperparameters use a scenario from the middle of the experimental grid - medium-sized with IR 9 (approximately).  

This phase of the study was highly manual, with hard-coded


Due to the large number of XGBoost hyperparameters, and to make tuing tractable, parameters are split into two groups and tuned separately. The first group is expected to be less sensitive to size and IR variation and includes:
* learning_rate
* subsample
* colsample_bytree
* reg_alpha
* reg_lamda

The remaining parameters include:
* min_child_weight
* max_depth
* gamma
* scale_pos_weight

Other parameters use defaults values:
* tree_method: hist
* objective: binary:logistic
* enable_categorical
* eval_metric: aucpr
* nthread: 4




## Read the full training and test sets

In [14]:
import pandas as pd

# Read the training and test sets
X_train_full = pd.read_parquet('..\\data\\processed\\X_train.parquet', engine='pyarrow')
X_test = pd.read_parquet('..\\data\\processed\\X_test.parquet', engine='pyarrow')

y_train_full = pd.read_parquet('..\\data\\processed\\y_train.parquet', engine='pyarrow')['TARGET']
y_test = pd.read_parquet('..\\data\\processed\\y_test.parquet', engine='pyarrow')['TARGET']

print(X_train_full.shape, X_test.shape, y_train_full.shape, y_test.shape)

(276759, 125) (30752, 125) (276759,) (30752,)


In [15]:
X_train_full

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_MISSING,EXT_SOURCE_1_MISSING,EXT_SOURCE_3_MISSING,OWN_CAR_AGE_MISSING,STATS_MISSING
ID,,,,,,,,,,,,,,,,,,,,,
371646,True,F,False,True,0,180000.0,781920.0,28215.0,675000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,1.0,False,True,False,True,True
139700,True,F,True,True,1,202500.0,1006920.0,42790.5,900000.0,"Spouse, partner",...,0.0,0.0,0.0,0.0,0.0,False,True,True,False,False
407492,True,F,False,False,0,36000.0,814041.0,23931.0,679500.0,Children,...,0.0,0.0,0.0,0.0,2.0,False,True,False,True,True
300400,True,F,True,False,1,135000.0,2013840.0,55507.5,1800000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,4.0,False,True,False,False,True
202199,True,F,False,False,0,157500.0,675000.0,65889.0,675000.0,Unaccompanied,...,0.0,0.0,1.0,0.0,1.0,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106567,True,F,False,True,0,180000.0,481495.5,32305.5,454500.0,Unaccompanied,...,0.0,0.0,0.0,0.0,4.0,False,True,False,True,True
297241,True,F,False,True,1,157500.0,846517.5,33700.5,684000.0,Unaccompanied,...,0.0,0.0,0.0,1.0,4.0,False,True,False,True,True
136325,False,F,False,True,1,135000.0,405000.0,20250.0,405000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True


## Imputer for numeric features

In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Create a preprocessor to replace missing with median for numeric columns only. Need to call
# 'set_output(transform='panda')' to ensure that the preprocessor works within GridSearchCV
numeric_cols = X_train_full.select_dtypes(include=['float64']).columns
preprocess = ColumnTransformer(
    transformers=[('median_imputer', SimpleImputer(strategy='median'), numeric_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

## First group

The implementation of the first group of hyperparameters is very manual because the run times were not initially know. Majority and minority class sizes are hard coded and manually updated for each scenario evaluated. Results are shown below the code cells. `scale_pos_weight`, which is directly related to IR, is also manually updated for each scenario.

### Create scenario

In [57]:
# Create tuning scenario from the middle of the experimental grid
n_maj, n_min = 21544, 798

# Separately sample majority and minority classes then combine them. Features and labels
X_tune_maj = X_train_full[y_train_full==0].sample(n_maj, random_state=56)
X_tune_min = X_train_full[y_train_full==1].sample(n_min, random_state=56)
X_tune = pd.concat([X_tune_maj, X_tune_min])

y_tune_maj = y_train_full[y_train_full==0].sample(n_maj, random_state=56)
y_tune_min = y_train_full[y_train_full==1].sample(n_min, random_state=56)
y_tune = pd.concat([y_tune_maj, y_tune_min])

print('Tune shape:', X_tune.shape, y_tune.shape)
print('n_maj:', n_maj, 'n_min', n_min)

Tune shape: (22342, 125) (22342,)
n_maj: 21544 n_min 798


In [44]:
X_tune

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_MISSING,EXT_SOURCE_1_MISSING,EXT_SOURCE_3_MISSING,OWN_CAR_AGE_MISSING,STATS_MISSING
ID,,,,,,,,,,,,,,,,,,,,,
240175,1,M,1,1,2,270000.0,643500.0,67702.5,643500.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,0
157354,1,F,0,0,2,247500.0,1054773.0,34857.0,945000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,0.0,0,1,0,1,1
386588,1,F,1,0,0,67500.0,545040.0,26640.0,450000.0,Family,...,0.0,0.0,0.0,1.0,0.0,0,1,0,0,1
112855,0,M,0,1,0,135000.0,270000.0,13500.0,270000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,1,0,1,1,1
183452,1,F,0,1,0,157500.0,323388.0,19386.0,292500.0,Unaccompanied,...,0.0,0.0,0.0,0.0,1.0,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296373,1,M,0,0,0,139500.0,675000.0,26154.0,675000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,1.0,0,1,0,1,1
142100,1,F,0,0,0,112500.0,545040.0,26640.0,450000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,2.0,0,1,0,1,1
152558,1,M,1,0,1,225000.0,675000.0,53329.5,675000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,1,1,1,0,1


### Define hyperparameter ranges

In [1]:
# Specify the fixed parameters and the range of values for tunable parameters

fixed_params = {
    # Fixed parameters
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'enable_categorical': True,
    'eval_metric': 'aucpr',             #['logloss', 'auc', 'aucpr']
    'nthread': 4,
    # Scenario-specific - initial selections for global tuning
    # Model complexity/Tree Structure
    'min_child_weight': 1,               # [1, 3, 5] 3-10
    'max_depth': 6,                      # [4, 6, 8] (reg)3-10
    # Regularisation
    'gamma': 1,                         # [0, 1, 5] (L2 reg) 0-5
    # Imbalance
    'scale_pos_weight': 27              # Maj/Min [x0.5, x1, x2]
}
tuning_params = {
    # Learning Rate
    'learning_rate': [0.03, 0.05, 0.1],
    # Sampling
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    # Regularisation
    'reg_alpha': [0, 0.5, 1],           #(L1 reg)0-1
    'reg_lambda': [0.5, 1, 2]           # 0-1]
}

### Create the training pipeline and grid search

In [59]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.pipeline import Pipeline

# Number of folds
cv=5

# Create the model and pipeline
tuning_model = XGBClassifier(**fixed_params)
global_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('clf', tuning_model)
])

# Convert the param to grid search-ready names e.g. clf__learning_rate
gs_tuning_params = {'clf__'+key: val for key, val in tuning_params.items()}

# Create the grid search based on the pipeline and pass the tuning params
global_grid = GridSearchCV(
    estimator=global_pipeline,
    param_grid=gs_tuning_params,
    scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
    #refit='pr_auc',
    refit=False,
    cv=cv,
    n_jobs=4,
    verbose=1
)
print(gs_tuning_params)

{'clf__learning_rate': [0.03, 0.05, 0.1], 'clf__subsample': [0.6, 0.8, 1.0], 'clf__colsample_bytree': [0.6, 0.8, 1.0], 'clf__reg_alpha': [0, 0.5, 1], 'clf__reg_lambda': [0.5, 1, 2]}


### Execute grid search

In [60]:
import time

# Exceute the grid search
start = time.perf_counter()
global_grid.fit(X_tune, y_tune)
runtime = (time.perf_counter() - start)/60

# Copied from Random Search
best_params_index = global_grid.cv_results_['mean_test_pr_auc'].argmax()
best_params = global_grid.cv_results_['params'][best_params_index]
best_pr_auc = global_grid.cv_results_['mean_test_pr_auc'][best_params_index].item()
best_roc_auc = global_grid.cv_results_['mean_test_roc_auc'][best_params_index].item()

print(f'Runtime: {runtime:.2f} min')
print('Sample size:', X_tune.shape[0], 'n_maj:', n_maj, 'n_min:', n_min)
print('Best pr_auc:', best_pr_auc)
print('Best roc_auc:', best_roc_auc)
print('Best params:', best_params)

print('--- Complete ---')

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 19.13 min
Sample size: 22342 n_maj: 21544 n_min: 798
Best pr_auc: 0.1072044542145166
Best roc_auc: 0.6842306129661279
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.03, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---


### Summary of hyperparameter tuning results

The following cells record the results of tuning on the first group of hyperparameters.

**Run with medium sample size, IR:27, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 19.81 min
Sample size: 44684 n_maj: 43088 n_min: 1596
Best pr_auc: 0.10386515655383988
Best roc_auc: 0.7104402454882718
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.03, 'clf__reg_alpha': 1, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with large sample size, IR:81, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 60.31 min
Sample size: 89368 n_maj: 88278 n_min: 1090
Best pr_auc: 0.04183232843757785
Best roc_auc: 0.6845372140820047
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 1, 'clf__reg_lambda': 0.5, 'clf__subsample': 0.6}
--- Complete ---
```

**Run with medium sample size, IR:81, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 47.81 min
Sample size: 44684 n_maj: 44139 n_min: 545
Best pr_auc: 0.044600588114199716
Best roc_auc: 0.6626191246485816
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 0.5, 'clf__reg_lambda': 2, 'clf__subsample': 1.0}
--- Complete ---
```

**Run with small sample size, IR:81, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 20.77 min
Sample size: 22342 n_maj: 22070 n_min: 272
Best pr_auc: 0.04104708221895009
Best roc_auc: 0.6401172119930616
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.1, 'clf__reg_alpha': 1, 'clf__reg_lambda': 0.5, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with large sample size, IR:27, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 36.09 min
Sample size: 89368 n_maj: 86176 n_min: 3192
Best pr_auc: 0.11113168473442037
Best roc_auc: 0.727196554331336
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.03, 'clf__reg_alpha': 0.5, 'clf__reg_lambda': 2, 'clf__subsample': 1.0}
--- Complete ---
```

**Run with small sample size, IR:27, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 19.13 min
Sample size: 22342 n_maj: 21544 n_min: 798
Best pr_auc: 0.1072044542145166
Best roc_auc: 0.6842306129661279
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.03, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with medimu sample size, IR:3, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 31.38 min
Sample size: 44684 n_maj: 33513 n_min: 11171
Best pr_auc: 0.5055575439394201
Best roc_auc: 0.7482321348428506
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 1, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with large sample size, IR:9, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 53.32 min
Sample size: 89368 n_maj: 80431 n_min: 8937
Best pr_auc: 0.26844393061069105
Best roc_auc: 0.7452050904510646
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 1.0}
--- Complete ---
```

**Run with small sample size, IR:1, and imputing by fold**
```
    Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 32.49 min
Sample size: 22342 n_maj: 11171 n_min: 11171
Best pr_auc: 0.7353302329436542
Best roc_auc: 0.7473264437063054
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 0.5, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with small sample size, IR:3, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 28.31 min
Sample size: 22343 n_maj: 16757 n_min: 5586
Best pr_auc: 0.5024488037954798
Best roc_auc: 0.7442434764214136
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 1, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```

**Run with medium sample size, IR:1, and imputing by fold**  
 ```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 20.15 min
Sample size: 44684 n_maj: 22342 n_min: 22342
Best pr_auc: 0.7420569951137401
Best roc_auc: 0.7510470081454461
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.1, 'clf__reg_alpha': 1, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
 ```

**Run with small sample size, IR:9, and imputing by fold**
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 15.15 min
Sample size: 22342 n_maj: 20108 n_min: 2234
Best pr_auc: 0.23798673783821114
Best roc_auc: 0.7203248152037651
Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}
--- Complete ---
```
 


**Run with large sample size, IR:3, and imputing by fold (with correct scale_pos_weight)**
```Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 37.90 min
Sample size: 89368 n_maj: 67026 n_min: 22342
Best pr_auc: 0.5128299915431205
Best roc_auc: 0.7538854875758638
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.1, 'clf__reg_alpha': 0.5, 'clf__reg_lambda': 1, 'clf__subsample': 1.0}
--- Complete ---
```

**Run with medium sample size, IR:9, and imputing by fold**  
```
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Runtime: 38.39 min
Sample size: 44684 n_maj: 40216 n_min: 4468
Best pr_auc: 0.25667169693466435
Best roc_auc: 0.7327876465760346
Best params: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 1.0}
--- Complete ---
```

In [76]:
print(global_grid.best_score_)
print(global_grid.best_params_)

0.3835886069551118
{'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.1, 'clf__reg_alpha': 0, 'clf__reg_lambda': 2, 'clf__subsample': 0.8}


In [80]:
runtime = (time.perf_counter() - start)/60

print(f'Runtime: {runtime:.2f} minutes')

Runtime: 7.83 minutes


In [64]:
print("Best params:", global_grid.best_params_)
print("Best AUC:", global_grid.best_score_)

Best params: {'colsample_bytree': 0.6, 'learning_rate': 0.05, 'reg_alpha': 0.5, 'reg_lambda': 2, 'subsample': 1.0}
Best AUC: 0.23598323267915747


## Second group

This section implements k-folds cross validation by scenario. For each scenario, XBGoost hyperparameters are tuned once on a unique train-validation split of unaugmented data, and used when executing both augmented and unaugmented experiments. k of 5 was chosen for small scenarios to ensure stable results, but reduced to 4 and 3 for larger scenario to manage runtime where stability was less of a concern. 

### Define hyperparameter ranges

In [2]:
# Set the fixed parameters and the grids for the tunable parameters

fixed_params = {
    # Fixed parameters
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'enable_categorical': True,
    'eval_metric': 'aucpr',
    'nthread': 1,

    # Tuned scenario-agnostic params
    'learning_rate': 0.05,
    # Sampling
    'subsample': 1.0,
    'colsample_bytree': 0.6,
    # Regularisation
    'reg_alpha': 0.5,              #L1 reg
    'reg_lambda': 2               #L2 reg
}
scenario_params = {
    # Scenario-specific - initial selections for global tuning
    # Model complexity/Tree Structure
    'min_child_weight': [1, 3, 5],
    'max_depth': [4, 6, 8],
    # Regularisation
    'gamma': [0, 1, 5],
    # Imbalance
    'scale_pos_weight': [0.5, 1, 2]         # Maj/Min [x0.5, x1, x2]
}

**Explanation of code**
Use pr_auc and pr_roc. PR is most relevant to unbalanced. RP_ROC improves stability.
Do not inclue F1. F1 is based on threshold of 0.5 which is arbitrary.

Returns only parameters and not the trained model. Therefore `refit=False`.



### Perform randomised CV search (`xgb_tune_params`)

In [221]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, TunedThresholdClassifierCV
from sklearn.metrics import make_scorer, f1_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
import time 

def xgb_tune_params(X, y):
    '''
    Use `RandomisedSearchCV` to tune the scenario-sensitive hyperparameters for XGBoost.
    '''
    # Calculate IR and the 'scale_pos_weight' param values based
    ir = y[y==0].shape[0] / y[y==1].shape[0]
    scenario_params['scale_pos_weight'] = [ir*0.5, ir, ir*2]
    print(tuning_params)

    # Choose the number of folds: large - 3; medium - 4; small - 5
    size = X.shape[0]
    if size > 80000:
        cv=3
    elif size > 40000:
        cv=4
    else:
        cv=5
    print(f'Using {cv} folds for size: {size:d}')

    # Create the estimator
    eval_model = XGBClassifier(**fixed_params)
    scenario_pipeline = Pipeline([
        ('preprocess', preprocess),
        ('clf', eval_model)
    ])
    
    # Convert the param to grid search-ready names e.g. clf__learning_rate
    gs_scenario_params = {'clf__'+key: val for key, val in scenario_params.items()}
    
    # Create the grid search based on the pipeline and pass the tuning params
    # Only the best parameters are required, so don't refit the final model
    scenario_grid = RandomizedSearchCV(
        estimator=scenario_pipeline,
        param_distributions=gs_scenario_params,
        scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
        refit=False,
        n_iter=30,
        cv=cv,
        n_jobs=4,
        verbose=1
    )
   
    # Perform the grid search
    start = time.perf_counter()
    scenario_grid.fit(X, y)   
    rs_runtime = (time.perf_counter() - start)/60

    # Return the results: best parameters; best pr_auc and roc_auc
    best_params_index = scenario_grid.cv_results_['mean_test_pr_auc'].argmax()
    best_params = scenario_grid.cv_results_['params'][best_params_index]
    best_pr_auc = scenario_grid.cv_results_['mean_test_pr_auc'][best_params_index].item()
    best_roc_auc = scenario_grid.cv_results_['mean_test_roc_auc'][best_params_index].item()

    # Strip the leading `clf__` characters from the param names
    best_params = {key.removeprefix('clf__'): val for key, val in best_params.items()}

    return {
        'n_folds': cv,
        'size': size,
        'ir': ir,
        'best_params': best_params,
        #'best_pr_auc_check': best_pr_auc_check,
        'best_pr_auc': best_pr_auc,
        'best_roc_auc': best_pr_auc,
        #'cv_results': scenario_grid.cv_results_['mean_test_pr_auc'][,
        #'best_index': scenario_grid.best_index_,
        'rs_runtime': rs_runtime
    } 
print('-- Complete --')

-- Complete --


### Create scenario data (`create_scenario`)

In [3]:
def create_scenario(X, y, size, ir):
    '''
    Create a random sample of specified size and imbalance ratio. This sample is specific to
    hyperparameter tuning, and independent of the sample used to evaluate baseline and 
    augmented classifier performance.
    '''

    # Calculate the number of majority and minority samples from the size and IR
    n_min = round(size / (1 + ir))
    n_maj = size - n_min

    # Take random samples of majority and minority class samples and concatenate them
    X_samp_maj = X[y==0].sample(n_maj, random_state=42)
    X_samp_min = X[y==1].sample(n_min, random_state=42)
    X_samp = pd.concat([X_samp_maj, X_samp_min])
    y_samp_maj = y[y==0].sample(n_maj, random_state=42)
    y_samp_min = y[y==1].sample(n_min, random_state=42)
    y_samp = pd.concat([y_samp_maj, y_samp_min])

    return X_samp, y_samp

In [4]:
# TEST
#X_samp, y_samp = create_scenario(X_train_full, y_train_full, 5000, 2)
#print('X_train shape:', X_train_full.shape[0], y_train_full.shape)
#print('X_samp:', X_samp.shape[0], y_samp.shape, 'n_maj:', y_samp[y_samp==0].shape, 'n_min:', y_samp[y_samp==1].shape)

### Execute tuning for all scenarios

In [224]:
import json, os

# Iterate through scenarios tuning hyperparameters and saving the results

scenarios = {
    'large': {'size': 89368, 'irs': [3, 9, 27, 81, 243]},
    'medium': {'size': 44684, 'irs': [1, 3, 9, 27, 81]},
    'small': {'size': 22342, 'irs': [1, 3, 9, 27, 81]}
}

# Iterate through the scenarios
for tier, params in scenarios.items():
    for ir in params['irs']:
        X_scenario, y_scenario = create_scenario(X_train_full, y_train_full, params['size'], ir)
        scenario_results = xgb_tune_params(X_scenario, y_scenario)

        path = '..\\scenarios\\parameters'
        filename = f'{tier}-ir{ir:03d}.json'
        fullpath = os.path.join(path, filename)
        with open(fullpath, 'w') as f:
            json.dump(scenario_results, f, indent=4)

        print(scenario_results)
print('--- Complete ---')

{'min_child_weight': 1, 'max_depth': 6, 'gamma': 1, 'scale_pos_weight': [9.989010989010989]}
Using 3 folds for size: 89368
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'n_folds': 3, 'size': 89368, 'ir': 3.0, 'best_params': {'scale_pos_weight': 1.5, 'min_child_weight': 5, 'max_depth': 8, 'gamma': 0}, 'best_pr_auc': 0.5112079271480527, 'best_roc_auc': 0.5112079271480527, 'rs_runtime': 4.208340215000013}
{'min_child_weight': 1, 'max_depth': 6, 'gamma': 1, 'scale_pos_weight': [9.989010989010989]}
Using 3 folds for size: 89368
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'n_folds': 3, 'size': 89368, 'ir': 8.999776211256574, 'best_params': {'scale_pos_weight': 4.499888105628287, 'min_child_weight': 5, 'max_depth': 6, 'gamma': 1}, 'best_pr_auc': 0.26877975295423784, 'best_roc_auc': 0.26877975295423784, 'rs_runtime': 4.419613223333727}
{'min_child_weight': 1, 'max_depth': 6, 'gamma': 1, 'scale_pos_weight': [9.989010989010989]}
Using 3 folds for size: 89368
Fit

### Tabulate the results

In [261]:
import json

# Read the params back from storage then tabulate them
params = ['scale_pos_weight', 'min_child_weight', 'max_depth', 'gamma']
tiers = ['large', 'medium', 'small']
irs = [1, 3, 9, 27, 81, 243]

best_params = {}
for tier, size_irs in scenarios.items():
    best_params[tier] = {}
    for ir in size_irs['irs']:
        path = '..\\scenarios\\parameters'
        filename = f'{tier}-ir{ir:03d}.json'
        fullpath = os.path.join(path, filename)
        with open(fullpath, "r") as f:
            scenario_params = json.load(f)
        best_params[tier][ir] = scenario_params['best_params']
#print(best_params)

# Tabulate the params
for p in params:
    print(f'Parameter: {p}')
    print('              1       3       9      27      81     243')
    print('       ================================================')
    for t in tiers:
        print(f'{t:7}', end='')
        for ir in irs:
            try:
                val = float(best_params[t][ir][p])
                print(f'{val:8.2f}', end='')
            except:
                print(f'      --', end='')
        print() 
    print()

Parameter: scale_pos_weight
              1       3       9      27      81     243
large        --    1.50    4.50   13.50  161.98  486.35
medium     0.50    1.50    4.50   13.50   40.49      --
small      2.00    1.50    4.50   13.50   81.14      --

Parameter: min_child_weight
              1       3       9      27      81     243
large        --    5.00    5.00    3.00    1.00    3.00
medium     5.00    5.00    5.00    5.00    3.00      --
small      5.00    3.00    5.00    1.00    3.00      --

Parameter: max_depth
              1       3       9      27      81     243
large        --    8.00    6.00    4.00    4.00    6.00
medium     8.00    6.00    6.00    4.00    4.00      --
small      6.00    4.00    4.00    4.00    4.00      --

Parameter: gamma
              1       3       9      27      81     243
large        --    0.00    1.00    1.00    5.00    0.00
medium     5.00    5.00    5.00    5.00    0.00      --
small      1.00    5.00    5.00    0.00    5.00      --



*<center>-- End of notebook --</center>*